<a href="https://colab.research.google.com/github/SriSharanya-617/RAG/blob/main/Langchain.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Langchain-build rag

In [4]:
!pip install -q langchain
!pip install -q langchain-community
!pip install -q langchain-core
!pip install -q langchain-ollama
!pip install -q faiss-cpu
!pip install -q sentence-transformers
!pip install -q pypdf
!pip install -q langchain-text-splitters

In [5]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceBgeEmbeddings
from langchain_community.llms import ollama

load pdf

In [6]:
loader=PyPDFLoader('employee_policy.pdf')
documents=loader. load()

print("Total pages", len(documents))
print(documents[0].page_content[:1000])

Total pages 1
Leave Policy Employees receive 12 casual leaves annually. Employees receive 15 sick 
leaves annually. Unused casual leaves cannot be carried forward. Work From Home 
Policy Employees may work from home twice per week. Manager approval is required 
for additional remote work. Travel Policy Travel expenses are reimbursed within 30 days. 
Original receipts must be submitted for reimbursement. Medical Insurance Policy All 
employees are covered under company medical insurance. Dependent coverage is 
available for spouse and children.


chunking

In [7]:
splitter=RecursiveCharacterTextSplitter(
chunk_size=500,
chunk_overlap=100
)

chunks=splitter.split_documents(documents)
print("Total Chunks:", len(chunks))
print(chunks[0].page_content)

Total Chunks: 2
Leave Policy Employees receive 12 casual leaves annually. Employees receive 15 sick 
leaves annually. Unused casual leaves cannot be carried forward. Work From Home 
Policy Employees may work from home twice per week. Manager approval is required 
for additional remote work. Travel Policy Travel expenses are reimbursed within 30 days. 
Original receipts must be submitted for reimbursement. Medical Insurance Policy All 
employees are covered under company medical insurance. Dependent coverage is


embedding

In [8]:
from langchain_community.embeddings import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
model_name='sentence-transformers/all-MiniLM-L6-v2'
)

/tmp/ipykernel_926/3762122373.py:3: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Create FAISS Vector Store


In [9]:
vector_store=FAISS.from_documents(
chunks,
embedding_model
)

print("FAISS Create Successfully")

FAISS Create Successfully


Create Retriever

In [10]:
retriever=vector_store.as_retriever(
search_type='similarity',
search_kwargs={'k':3}
)

print("Retriver Ready")

Retriver Ready


Test Retrieval

In [12]:
query = "What is machine learning?"

docs = retriever.invoke(query)

for doc in docs:
    print(doc.page_content)
    print("=" * 50)

employees are covered under company medical insurance. Dependent coverage is 
available for spouse and children.
Leave Policy Employees receive 12 casual leaves annually. Employees receive 15 sick 
leaves annually. Unused casual leaves cannot be carried forward. Work From Home 
Policy Employees may work from home twice per week. Manager approval is required 
for additional remote work. Travel Policy Travel expenses are reimbursed within 30 days. 
Original receipts must be submitted for reimbursement. Medical Insurance Policy All 
employees are covered under company medical insurance. Dependent coverage is


Create Context

In [14]:
query = "What is machine learning?"

docs = retriever.invoke(query)

context = "\n".join([doc.page_content for doc in docs])

print(context)

employees are covered under company medical insurance. Dependent coverage is 
available for spouse and children.
Leave Policy Employees receive 12 casual leaves annually. Employees receive 15 sick 
leaves annually. Unused casual leaves cannot be carried forward. Work From Home 
Policy Employees may work from home twice per week. Manager approval is required 
for additional remote work. Travel Policy Travel expenses are reimbursed within 30 days. 
Original receipts must be submitted for reimbursement. Medical Insurance Policy All 
employees are covered under company medical insurance. Dependent coverage is


Connect LLM

In [16]:
from transformers import pipeline

llm = pipeline(
    "text-generation",
    model="google/flan-t5-base"
)

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

[transformers] The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'Cohere2MoeForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'CwmForCausalLM', 'Data2VecTextForCausalLM', 'DbrxForCausalLM', 'DeepseekV2ForCausalLM', 'DeepseekV3ForCausalLM', 'DeepseekV4ForCausalLM', 'DiffLlamaForCausalLM', 'DogeForCausalLM', 'Dots1ForCausalLM', 'ElectraForCausalLM', 'Emu3ForCausalLM', 'ErnieForCausalLM', 'Ernie4_5ForCausalLM', 'Ernie4_5_MoeForCausalLM', 'Exaone4ForCausalLM', 'ExaoneMoeForCausal

Create RAG Function

In [19]:
def rag(query):

    docs = retriever.invoke(query)

    context = "\n".join(
        [doc.page_content for doc in docs]
    )

    prompt = f"""
Answer the question using only the context.

Context:
{context}

Question:
{query}

Answer:
"""

    result = llm(
        prompt,
        max_new_tokens=100
    )

    return result[0]["generated_text"]

Ask Questions

In [20]:
question = "What is machine learning?"

answer = rag(question)

print(answer)

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Answer the question using only the context.

Context:
employees are covered under company medical insurance. Dependent coverage is 
available for spouse and children.
Leave Policy Employees receive 12 casual leaves annually. Employees receive 15 sick 
leaves annually. Unused casual leaves cannot be carried forward. Work From Home 
Policy Employees may work from home twice per week. Manager approval is required 
for additional remote work. Travel Policy Travel expenses are reimbursed within 30 days. 
Original receipts must be submitted for reimbursement. Medical Insurance Policy All 
employees are covered under company medical insurance. Dependent coverage is

Question:
What is machine learning?

Answer:

